In [1]:
import os

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockLatestTradeRequest
from alpaca.data.enums import DataFeed


client = StockHistoricalDataClient(
    os.environ["ALPACA_API_KEY"],
    os.environ["ALPACA_SECRET_KEY"]
)

request = StockLatestTradeRequest(
    symbol_or_symbols=["AAPL"],
    feed=DataFeed.IEX
)

trades = client.get_stock_latest_trade(request)

trade = trades["AAPL"]

print("price:", trade.price)
print("size:", trade.size)
print("timestamp:", trade.timestamp)

price: 309.895
size: 70.0
timestamp: 2026-08-25 19:59:58.123484+00:00


In [31]:
import os
from datetime import datetime, timedelta, timezone

from dash import Dash, dcc, html
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.enums import DataFeed


SYMBOL = "PLTR"


# =========================
# 1. Alpaca client
# =========================

client = StockHistoricalDataClient(
    os.environ["ALPACA_API_KEY"],
    os.environ["ALPACA_SECRET_KEY"]
)


# =========================
# 2. 获取历史K线
# =========================

request = StockBarsRequest(
    symbol_or_symbols=[SYMBOL],
    timeframe=TimeFrame.Minute,
    start=datetime.now(timezone.utc) - timedelta(days=1),
    end=datetime.now(timezone.utc) - timedelta(minutes=20),
    feed=DataFeed.IEX
)

bars = client.get_stock_bars(request)

In [32]:
df = bars.df.xs(SYMBOL).copy()

# 计算均线
df["ma20"] = df["close"].rolling(20).mean()
df["ma60"] = df["close"].rolling(60).mean()

# 计算 MACD (12, 26, 9)
ema12 = df["close"].ewm(span=12, adjust=False).mean()
ema26 = df["close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
df["macd_hist"] = df["macd"] - df["macd_signal"]

# 保留最近 200 根
# df = df.tail(400)

In [33]:
# =========================
# 3. 创建 subplot
# =========================

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,

    # 价格 / 成交量 / MACD
    row_heights=[0.55, 0.1, 0.25]
)


# =========================
# 4. 蜡烛图
# =========================

fig.add_trace(
    go.Candlestick(
        x=df.index,
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name=SYMBOL,
        legend="legend"
    ),
    row=1,
    col=1
)


# =========================
# 5. MA20
# =========================

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["ma20"],
        mode="lines",
        name="MA20",
        line=dict(width=1.5),
        legend="legend"
    ),
    row=1,
    col=1
)


# =========================
# 6. MA60
# =========================

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["ma60"],
        mode="lines",
        name="MA60",
        line=dict(width=1.5),
        legend="legend"
    ),
    row=1,
    col=1
)


# =========================
# 7. 成交量
# =========================

fig.add_trace(
    go.Bar(
        x=df.index,
        y=df["volume"],
        name="Volume",
        showlegend=False
    ),
    row=2,
    col=1
)


# =========================
# 8. MACD
# =========================

hist_colors = ["#26a69a" if v >= 0 else "#ef5350" for v in df["macd_hist"]]

fig.add_trace(
    go.Bar(
        x=df.index,
        y=df["macd_hist"],
        name="MACD Hist",
        marker_color=hist_colors,
        showlegend=False
    ),
    row=3,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["macd"],
        mode="lines",
        name="MACD",
        line=dict(width=1.5),
        legend="legend2"
    ),
    row=3,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["macd_signal"],
        mode="lines",
        name="Signal",
        line=dict(width=1.5),
        legend="legend2"
    ),
    row=3,
    col=1
)


# =========================
# 9. Layout
# =========================

fig.update_layout(
    title=f"{SYMBOL} - 1 Minute",
    height=1000,

    # 移除 Plotly 默认的 range slider
    xaxis_rangeslider_visible=False,

    hovermode="x unified",

    margin=dict(
        l=60,
        r=120,
        t=70,
        b=50
    ),

    # 价格子图右侧：AAPL / MA20 / MA60
    legend=dict(
        x=1.01,
        y=1,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.7)"
    ),

    # MACD 子图右侧：MACD / Signal
    legend2=dict(
        x=1.01,
        y=0.25,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.7)"
    )
)


fig.update_yaxes(
    title_text="Price",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Volume",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="MACD",
    row=3,
    col=1
)

fig.update_xaxes(
    title_text="Time",
    row=3,
    col=1
)

# 按各子图实际高度把 legend 贴到对应图右侧
fig.update_layout(
    legend=dict(y=fig.layout.yaxis.domain[1]),
    legend2=dict(y=fig.layout.yaxis3.domain[1])
)


# =========================
# 10. Dash
# =========================

app = Dash(__name__)

app.layout = html.Div(
    [
        html.H2(
            f"{SYMBOL} Market Data",
            style={"textAlign": "center"}
        ),

        dcc.Graph(
            id="candlestick-chart",
            figure=fig,
            style={
                "width": "100%"
            }
        )
    ]
)


if __name__ == "__main__":
    app.run(debug=True)